# 03 — Layer-wise SAE Training on PIE

## Objective

Train a separate TopK sparse autoencoder (SAE) on the activations of **all 14 PIE Transformer blocks**, using the original `TopKSAE` and `SAETrainer` classes in Kendiukhov's `bio-sae` repository.

The analysis uses 14 layer-specific SAE dictionaries initialized with distinct seeds. Checkpoint paths and training settings are recorded in an independent-run manifest.

**Reference implementation:** `external/bio-sae/src/sae_model.py`, `02_train_sae.py`, `02b_train_all_layers.py`. PIE-specific extraction is performed by notebook 02. The reusable training adapter is `src/sae_training.py`.

**Run behavior:** this notebook checks the configuration of existing completed runs and reuses them; a missing layer is trained and saved, while incomplete or incompatible existing runs cause an error. No old checkpoint is overwritten.


## 1. Environment and input validation

The two HDF5 datasets contain **128 training and 32 validation perturbation conditions** from notebook 02. We use the same conditions for all 14 layers. Encoder/processor vectors represent 512 latent positions per condition; decoder vectors represent 6642 gene positions. The latter are **not 850,176 independent biological experiments**.

The input activation vectors have dimension 768. All centering statistics are calculated on the training split only; the separate validation split is not used for parameter optimization.


In [1]:
from pathlib import Path
import csv
import json
import subprocess
import sys
import hashlib

import h5py
import numpy as np
import torch

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "sae_training.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "src" / "sae_training.py").is_file(), PROJECT_ROOT
assert (PROJECT_ROOT / "external" / "bio-sae" / "src" / "sae_model.py").is_file()

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from sae_training import train_sae_layer

DATA_DIR = PROJECT_ROOT / "data" / "activations" / "expanded_128_32"
DATA_SPECS = {
    "train": (DATA_DIR / "train_128.h5", 128),
    "validation": (DATA_DIR / "val_32.h5", 32),
}
LAYER_COUNTS = {"encoder": 4, "processor": 6, "decoder": 4}
POSITIONS = {"encoder": 512, "processor": 512, "decoder": 6642}

for split_name, (path, n_conditions) in DATA_SPECS.items():
    assert path.is_file(), path
    with h5py.File(path, "r") as f:
        assert f.attrs["split"] == split_name
        assert int(f.attrs["n_conditions"]) == n_conditions
        assert int(f.attrs["completed_count"]) == n_conditions
        assert f["condition_indices"].shape == (n_conditions,)
        assert f["gene_ids"].shape == (n_conditions, 6642)
        for component, count in LAYER_COUNTS.items():
            for index in range(count):
                ds = f[component][str(index)]
                assert ds.shape == (n_conditions, POSITIONS[component], 768)
                assert ds.dtype == np.float32
    print(f"Validated {split_name} metadata: {n_conditions} conditions, 14 layers")

BIO_SAE_COMMIT = subprocess.check_output(
    ["git", "-C", str(PROJECT_ROOT / "external" / "bio-sae"), "rev-parse", "HEAD"],
    text=True,
).strip()
print("bio-sae commit:", BIO_SAE_COMMIT)
print("Training adapter:", SRC_DIR / "sae_training.py")


Validated train metadata: 128 conditions, 14 layers
Validated validation metadata: 32 conditions, 14 layers
bio-sae commit: 3ac41b1e7725c224dc1225a3311e8802eb96da85
Training adapter: D:\bio_paper\pie-sae\src\sae_training.py


## 2. Protocol selection: original reference vs. PIE adaptation

The original bulk training code uses TopK=32, dictionary expansion ×4, Adam with learning rate `3e-4`, **5 epochs** and **batch size 4096**. The PIE-adapted protocol retains the original model, trainer, loss and normalization implementation, but uses **20 epochs** and **batch size 256**.

The adaptation was checked using the **same initialization seed within each comparison**, on two representation types:

| Layer | Initialization seed | Reference: 5 epochs / 4096, validation R² | Adapted: 20 epochs / 256, validation R² |
|:--|--:|--:|--:|
| `encoder.1` | 1001 | 0.8182 | **0.9229** |
| `decoder.3` | 1013 | 0.7742 | **0.8275** |

Supporting results are stored under `runs/sae_reference_protocol/` and `runs/sae_protocol_tests/`. The reference configuration was a **control**, not the final training configuration.

**Interpretation:** the combined change in epochs, batch size and optimizer-update count improves reconstruction on both tested layers. These controls do *not* isolate the effects of epochs versus batch size, or establish biological interpretability. They are not a formal hyperparameter search. Unlike the original implementation's training-sample evaluation, all reported PIE validation metrics are computed on the held-out conditions.

The reported feature analyses use only the independently initialized checkpoints in `runs/sae_protocol_tests/`.


## 3. Final training: 14 independently initialized SAEs

- Encoder: `encoder.0`–`encoder.3`; processor: `processor.0`–`processor.5`; decoder: `decoder.0`–`decoder.3`.
- Each SAE has input dimension 768, 3072 dictionary features, and TopK=32.
- Hyperparameters: 20 epochs, batch size 256, Adam learning rate `3e-4`.
- Initialization seeds are **1000–1013** in layer order; the minibatch-order seed is **42** for every layer.
- Training and validation are kept separate; a training-only mean is stored per layer.
- Checkpoints and JSON metadata are saved to `runs/sae_protocol_tests/` and the aggregate CSV to `results/sae_training/`.

Separate initialization seeds avoid imposing an identical initialization across different layers. This design does not measure feature stability under repeated fitting of an individual layer.


In [2]:
LAYERS = (
    [("encoder", i) for i in range(4)]
    + [("processor", i) for i in range(6)]
    + [("decoder", i) for i in range(4)]
)
assert len(LAYERS) == 14

EPOCHS = 20
BATCH_SIZE = 256
LEARNING_RATE = 3e-4
ORDER_SEED = 42
RUN_GROUP = "sae_protocol_tests"  # Existing final-series checkpoints live here.

final_results = []
for number, (component, index) in enumerate(LAYERS):
    layer_name = f"{component}.{index}"
    init_seed = 1000 + number
    print(f"\n[{number + 1}/14] {layer_name}, initialization seed={init_seed}")
    result = train_sae_layer(
        project_root=PROJECT_ROOT,
        component=component,
        layer_index=index,
        init_seed=init_seed,
        order_seed=ORDER_SEED,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        run_group=RUN_GROUP,
    )
    metrics = result["final_validation"]
    assert result["complete"] and result["bio_sae_commit"] == BIO_SAE_COMMIT
    assert result["config"]["init_seed"] == init_seed
    assert result["config"]["layer"] == layer_name
    assert abs(metrics["l0"] - 32.0) < 1e-5
    assert np.isfinite(metrics["r2"])
    expected_steps = (5120 if component != "decoder" else 66420)
    assert result["optimizer_steps"] == expected_steps
    final_results.append({
        "layer": layer_name,
        "init_seed": init_seed,
        "val_mse": metrics["mse"],
        "val_r2": metrics["r2"],
        "explained_variance": metrics["explained_variance"],
        "l0": metrics["l0"],
        "inactive_features": metrics["inactive_features"],
        "optimizer_steps": result["optimizer_steps"],
        "checkpoint": result["checkpoint"],
    })

assert len(final_results) == 14
assert len({row["init_seed"] for row in final_results}) == 14

SUMMARY_DIR = PROJECT_ROOT / "results" / "sae_training"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
summary_path = SUMMARY_DIR / "independent_sae_summary.csv"
with summary_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(final_results[0]))
    writer.writeheader()
    writer.writerows(final_results)

print("\nFinal validation summary:")
print(f"{'Layer':<14}{'Init':>7}{'R²':>10}{'L0':>8}{'Inactive':>11}")
for row in final_results:
    print(f"{row['layer']:<14}{row['init_seed']:>7}{row['val_r2']:>10.4f}"
          f"{row['l0']:>8.2f}{row['inactive_features']:>11}")
print("\nAll 14 completed. CSV:", summary_path)



[1/14] encoder.0, initialization seed=1000
Verified existing SAE, skipping training: encoder.0
Validation R²: 0.9821

[2/14] encoder.1, initialization seed=1001
Verified existing SAE, skipping training: encoder.1
Validation R²: 0.9229

[3/14] encoder.2, initialization seed=1002
Verified existing SAE, skipping training: encoder.2
Validation R²: 0.8820

[4/14] encoder.3, initialization seed=1003
Verified existing SAE, skipping training: encoder.3
Validation R²: 0.8473

[5/14] processor.0, initialization seed=1004
Verified existing SAE, skipping training: processor.0
Validation R²: 0.8440

[6/14] processor.1, initialization seed=1005
Verified existing SAE, skipping training: processor.1
Validation R²: 0.8340

[7/14] processor.2, initialization seed=1006
Verified existing SAE, skipping training: processor.2
Validation R²: 0.8256

[8/14] processor.3, initialization seed=1007
Verified existing SAE, skipping training: processor.3
Validation R²: 0.8224

[9/14] processor.4, initialization seed

## 4. Checkpoint integrity and provenance

The reusable trainer checks matching configuration and the presence of completed files before reusing a run. The following independent read verifies model tensor shapes and finiteness, the saved preprocessing mean, result metadata and the pinned upstream code revision. It also writes a compact checkpoint manifest with SHA-256 digests. This check does not repeat training or PIE inference.


In [3]:
def sha256_file(path: Path) -> str:
    hasher = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

checkpoint_records = []
for row in final_results:
    checkpoint_path = PROJECT_ROOT / row["checkpoint"]
    run_dir = checkpoint_path.parent
    result_file = run_dir / "results.json"
    mean_file = run_dir / "activation_mean.npy"
    assert all(p.is_file() for p in [checkpoint_path, result_file, mean_file])

    saved_result = json.loads(result_file.read_text(encoding="utf-8"))
    saved_ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    assert saved_result["complete"] and saved_result["bio_sae_commit"] == BIO_SAE_COMMIT
    assert saved_ckpt["config"] == saved_result["config"]
    assert saved_result["config"]["init_seed"] == row["init_seed"]
    weights = saved_ckpt["model_state_dict"]
    assert weights["W_dec.weight"].shape == (768, 3072)
    assert all(torch.isfinite(t).all().item() for t in weights.values())
    mean = np.load(mean_file)
    assert mean.shape == (768,) and np.isfinite(mean).all()

    record = {
        "layer": row["layer"],
        "checkpoint": row["checkpoint"],
        "checkpoint_sha256": sha256_file(checkpoint_path),
        "init_seed": row["init_seed"],
        "validation_r2": row["val_r2"],
    }
    checkpoint_records.append(record)
    print(f"Verified {row['layer']}: {checkpoint_path.name}")

manifest = {
    "series": "PIE-SAE independent layer-wise",
    "bio_sae_commit": BIO_SAE_COMMIT,
    "training_module": "src/sae_training.py",
    "training_config": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "order_seed": ORDER_SEED,
        "initialization_seeds": list(range(1000, 1014)),
        "normalization": "original_dim1",
        "topk": 32,
        "n_features": 3072,
    },
    "input_extraction_manifest": "results/extraction_validation/extraction_manifest.json",
    "checkpoints": checkpoint_records,
}
manifest_path = SUMMARY_DIR / "independent_sae_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
assert len(checkpoint_records) == 14
print("\nAll checkpoints verified. Manifest:", manifest_path)


Verified encoder.0: sae_final.pt
Verified encoder.1: sae_final.pt
Verified encoder.2: sae_final.pt
Verified encoder.3: sae_final.pt
Verified processor.0: sae_final.pt
Verified processor.1: sae_final.pt
Verified processor.2: sae_final.pt
Verified processor.3: sae_final.pt
Verified processor.4: sae_final.pt
Verified processor.5: sae_final.pt
Verified decoder.0: sae_final.pt
Verified decoder.1: sae_final.pt
Verified decoder.2: sae_final.pt
Verified decoder.3: sae_final.pt

All checkpoints verified. Manifest: D:\bio_paper\pie-sae\results\sae_training\independent_sae_manifest.json


## 5. Results and conclusions

**Final independently initialized series:** all 14 SAEs completed successfully. The recorded validation R² ranges from approximately **0.8119** (`processor.5`) to **0.9821** (`encoder.0`). At the selected TopK=32 setting, mean L0 is 32 in every layer, and no feature is inactive on the evaluated validation positions in this series.

The strong reconstruction is evidence that the SAE dictionaries can represent the sampled PIE activations with the chosen sparsity budget. It does **not** demonstrate that individual features encode biological concepts, regulatory mechanisms or stable cross-layer identities.

**Methodological differences from Kendiukhov:** the model is PIE rather than Geneformer/scGPT; hidden dimension is 768, decoder and latent-slot observations have different meanings, the data consist of 128 training and 32 validation perturbation conditions, and the PIE-adapted protocol uses 20 epochs / batch 256 instead of 5 epochs / batch 4096. The primary SAEs were trained with different fixed initialization seeds. The code uses the upstream `TopKSAE` and `SAETrainer` implementations unchanged.

**Downstream provenance.** Notebooks 04 and 05 load the fixed checkpoints listed in `independent_sae_manifest.json` under `runs/sae_protocol_tests/`.

**Limitations:** each SAE has been trained once at a layer-specific seed; independent seeds across layers address one initialization confound but do not establish feature stability across repeated fits of the *same* layer. The 6642 gene vectors per condition are not independent perturbation experiments. Subsequent analysis requires biological annotations and appropriate statistical controls.
